# Canopy Photosynthesis Modelling Practical
**English edition · 60 minutes · three experimental modules**

Read the principles, run a complete example, write a short piece of Python, check it,
and explain the result. Geometry generation, triangle clipping, C/OpenMP ray tracing
and plotting tools are supplied so that you can focus on equations and controlled comparisons.

| Module | Programming task | Main output |
|---|---|---|
| [1. Structure and central cropping](01_Structure_and_Cropping.ipynb) | Implement triangle area and LAI; change leaf angles | Curved maize leaves and the central simulation cell |
| [2. Light and daily integration](02_Light_and_Daily_Integration.ipynb) | Convert photon packets to PPFD; implement trapezoidal integration | Facet light distribution and daily photon balance |
| [3. Photosynthesis and design](03_Photosynthesis_and_Design.ipynb) | Implement net A-Q and area weighting; calculate treatment contrasts | Daily CO2 uptake and canopy-design comparison |

[Runnable worked solutions](04_Worked_Solutions.ipynb)


In [ ]:
# Run this preparation cell once after starting/restarting the kernel.
# Finds normal, flattened, nested and ZIP-preserved course uploads.
from pathlib import Path
import sys,os,subprocess,hashlib,zipfile,shutil

def locate_canopy_runtime(input_root=None,local_roots=None,work=None,tag=None):
    tag=tag or ('win_amd64' if sys.platform=='win32' else 'manylinux_2_34_x86_64')
    expected=f'canopyteach-1.0.1-cp311-abi3-{tag}.whl'
    pattern=f'canopyteach-1.0.1-cp311-abi3-{tag}*.whl'
    work=Path(work or '/kaggle/working/canopyteach_practical_en')
    candidate_root=Path(input_root or '/kaggle/input')
    wheels=set();helpers=set();archives=set();observed=[];archive_errors=[]

    def consider(p):
        if p.name=='canopy_practical.py':helpers.add(p.resolve())
        elif p.match(pattern):wheels.add(p.resolve())
        elif p.suffix.lower()=='.zip':archives.add(p.resolve())
        if len(observed)<35 and p.is_file():observed.append(str(p))

    if candidate_root.is_dir():
        # Kaggle can add dataset/owner/version directories or flatten uploads.
        for p in candidate_root.rglob('*'):
            if p.is_file():consider(p)
    else:
        roots=local_roots if local_roots is not None else ([] if input_root is not None else [Path.cwd(),*Path.cwd().parents])
        for folder in map(Path,roots):
            if folder==Path(folder.anchor):continue
            for p in [folder/'canopy_practical.py',*folder.glob(pattern),*folder.glob('wheels/'+pattern)]:
                if p.is_file():consider(p)
        if input_root is not None:
            observed.append('Input directory does not exist: '+str(candidate_root))

    # Read only the runtime members of course ZIPs into a writable directory.
    # Never extract arbitrary archive paths or change the input Dataset.
    if not wheels or not helpers:
        for archive in sorted(archives):
            try:
                with zipfile.ZipFile(archive) as z:
                    selected=[n for n in z.namelist() if not n.endswith('/') and
                        (Path(n).name=='canopy_practical.py' or Path(n).match(pattern))]
                    if not selected:continue
                    key=hashlib.sha256(str(archive).encode()).hexdigest()[:12]
                    for i,name in enumerate(selected):
                        info=z.getinfo(name)
                        if info.file_size>30_000_000:raise ValueError('Runtime member exceeds 30 MB: '+name)
                        dest=work/'_runtime'/key/str(i)/Path(name).name
                        dest.parent.mkdir(parents=True,exist_ok=True)
                        dest.write_bytes(z.read(name));consider(dest)
            except (zipfile.BadZipFile,OSError,ValueError) as exc:
                archive_errors.append(str(archive)+': '+str(exc))

    if not wheels or not helpers:
        missing=[]
        if not wheels:missing.append(expected)
        if not helpers:missing.append('canopy_practical.py')
        details='\n'.join(observed) or '(no mounted input files found)'
        raise FileNotFoundError(
            'Missing course runtime: '+', '.join(missing)+'\n'
            'Add the course Dataset as an input to THIS notebook; importing an .ipynb alone does not attach its Dataset.\n'
            'Searched input: '+str(candidate_root)+'\nObserved files (first 35):\n'+details+
            ('\nArchive errors:\n'+'\n'.join(archive_errors) if archive_errors else ''))

    # Prefer a nearby helper, without requiring the wheels/ directory layout.
    pairs=[]
    for wheel in sorted(wheels):
        for helper in sorted(helpers):
            shared=len(os.path.commonpath([str(wheel.parent),str(helper.parent)]).split(os.sep))
            nearby=helper.parent in (wheel.parent,wheel.parent.parent)
            pairs.append((not nearby,-shared,str(wheel),str(helper)))
    _,_,wheel_name,helper_name=min(pairs)
    wheel,helper=Path(wheel_name),Path(helper_name)
    # Browser downloads may add a suffix such as " (1)" to a wheel filename.
    if wheel.name!=expected:
        normalized=work/'_runtime'/'normalized'/expected
        normalized.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(wheel,normalized);wheel=normalized
    return wheel,helper

if __name__=='__main__':
    if sys.version_info<(3,11):raise RuntimeError('This course requires CPython 3.11 or newer.')
    input_override=os.environ.get('CANOPYTEACH_INPUT_ROOT')
    work_override=os.environ.get('CANOPYTEACH_WORK_ROOT')
    on_kaggle=Path('/kaggle/working').is_dir()
    WORK=Path(work_override) if work_override else (
        Path('/kaggle/working/canopyteach_practical_en') if on_kaggle else Path.cwd()/'results/practical_en')
    wheel,helper=locate_canopy_runtime(input_root=input_override,work=WORK)
    PACKAGE_ROOT=helper.parent
    WORK.mkdir(parents=True,exist_ok=True);site=WORK/'_packages';marker=site/'course_wheel.sha256'
    signature=hashlib.sha256(wheel.read_bytes()).hexdigest()
    if not marker.exists() or marker.read_text().strip()!=signature:
        if 'canopyteach' in sys.modules:
            raise RuntimeError('Restart the kernel before installing a different course wheel, then run preparation again.')
        subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--disable-pip-version-check',
                        '--upgrade','--target',str(site),str(wheel)],check=True)
        marker.write_text(signature)
    sys.path.insert(0,str(site));sys.path.insert(0,str(PACKAGE_ROOT))
    import canopy_practical as cp
    import canopyteach
    if canopyteach.__version__!='1.0.1':raise RuntimeError('Unexpected canopyteach version; restart the kernel.')
    cp.style()
    try:get_ipython().run_line_magic('matplotlib','inline')
    except NameError:pass
    DEFAULT_STEP=int(os.environ.get('CANOPYTEACH_STEP_MINUTES','30'))
    DEFAULT_SPACING=float(os.environ.get('CANOPYTEACH_RAY_SPACING','.005'))
    print('Runtime wheel:',wheel)
    print('Teaching helper:',helper)
    print('Ready. Outputs:',WORK)
    print('Random whole-plant azimuth; central crop; XY periodic; C/OpenMP; synthetic direct light.')


## Learning objectives, schedule and operation

| Time | Activity | Principles / example / exercises / discussion |
|---|---|---|
| 0–5 min | Introduction and execution check | Identify inputs, outputs and units |
| 5–20 min | Module 1 | 4 / 3 / 6 / 2 min |
| 20–40 min | Module 2 | 5 / 4 / 8 / 3 min |
| 40–60 min | Module 3 | 4 / 4 / 9 / 3 min |

**Prerequisites:** Python variables, arrays and simple functions. The complex maize
geometry and C ray tracer are provided; you do not have to implement them in this practical.

**Operation:** select a code cell and press **Shift+Enter**; save with **Ctrl+S**.
After starting or restarting a kernel, run the preparation cell near the top of that notebook.
Each experiment then imports its dependencies and creates its own inputs: you can run an
example, exercise, parameter experiment or reference without first running another experiment.
`return None` marks a task to complete. Unfinished exercises display guidance rather than
an intentional exception. Reference functions keep their variables local and preserve your work.

Saved figures are examples from actual execution, not evidence that your current kernel
has run a cell. Restart the kernel after editing a Python module.


## Reference conditions and scope

Generate a larger 7 x 7 stand, then spatially clip its central 3 x 3 ground cell.
Keep leaf fragments entering from neighboring plants, including plants whose stems are outside.
Apply XY-periodic ray boundaries to the clipped central cell; do not wrap all 49 complete plants
into that cell. A 9 x 9 source-stand check gave the same central geometry and daily PPFD.

The model preserves the supplied maize leaf width, natural curvature and edge waves.
Direct light uses C/OpenMP forward energy packets. The hit density already contains the
projection effect; an additional leaf-normal cosine is not applied.

Date 2026-06-21; latitude 31 degrees N, longitude 121 degrees E, UTC+8.
Daytime direct-normal PPFD (DNI) is 1500; A-Q physiology is fixed.
Architecture, weather and physiology are synthetic teaching conditions, not field measurements.
The classroom model excludes sky diffuse light, multiple scattering and biochemical C4 dynamics.

**Whole-plant azimuth:** each plant has one seeded uniform 0–360° rotation about Z.
Its original 180° within-plant leaf sequence is retained. The same central-cell angle map
is repeated into the source buffer and across periodic boundaries. Seed 20261025 makes
comparisons reproducible; keep that realization fixed when comparing leaf angles.
The former all-aligned canopy is retained only as a labelled diagnostic control.


In [ ]:
import matplotlib.pyplot as plt
import canopy_practical as cp
from canopyteach import CanopyLight,solar_position
from canopyteach.simulation import prepare_canopy
mesh=prepare_canopy();sp=solar_position('2026-06-21',12,31,121,8)
light=CanopyLight(mesh,ray_spacing=DEFAULT_SPACING,threads=4).trace(sp['sun'],1500.)
cp.print_geometry(cp.geometry_trial()[1])
print('Execution successful; threads:',light.stats['threads_used'])
print('Leaf/stem/escape fractions:',light.fractions)
fig=cp.crop_figure();plt.show()


## Synthesis and submission

Save your three module notebooks. Submit the completed area/LAI expressions, daily-integral
self-check, two selected leaf angles, daily photon and CO2 contrasts, and one interpretation.
Also keep `module3_design/experiment.json` from the working directory.

An unsuccessful design is still a valid submission if the comparison, calculation and
explanation are correct. Do not change density or physiology while attributing a result to angles.
Reference values use 30-minute time steps and a 5 mm ray raster. If settings change, use your
calculated outputs rather than forcing agreement with these default values.

The source is in `csrc/`, `src/canopyteach/` and `canopy_practical.py` in the full project.
